In [0]:
from pyspark.sql import functions as F

full_df = spark.table("banking_contracts_demo.bronze.accounts_daily")

latest_batch_id = (
    full_df.select("batch_id")
           .distinct()
           .orderBy(F.col("batch_id").desc())
           .limit(1)
           .collect()[0]["batch_id"]
)

df = full_df.filter(F.col("batch_id") == latest_batch_id)
total_records = df.count()

print(f"Validating batch: {latest_batch_id}")
print(f"Records in this batch: {total_records}")

In [0]:
from pyspark.sql import functions as F

results = []

def mask_ssn(col):
    return F.when(col.isNull(), F.lit(None)).otherwise(
        F.concat(F.lit("***-**-"), F.substring(col, -4, 4))
    )

def record_check(name, violating_df, description):
    count = violating_df.count()
    status = "PASS" if count == 0 else "FAIL"
    sample = violating_df.select(
        "account_id",
        mask_ssn(F.col("customer_ssn")).alias("customer_ssn_masked"),
        "customer_name", "balance", "account_type"
    ).limit(5)
    results.append({
        "check_name": name,
        "description": description,
        "status": status,
        "violation_count": count,
    })
    print(f"[{status}] {name} -- {count} violation(s)")
    if count > 0:
        sample.show(truncate=False)

In [0]:
results = []

# 1. account_id: not null
record_check(
    "account_id_not_null",
    df.filter(F.col("account_id").isNull()),
    "account_id must not be null"
)

# 2. account_id: unique — check against ALL history
dupe_ids_all_time = (
    full_df.groupBy("account_id").count().filter("count > 1").select("account_id")
)
record_check(
    "account_id_unique",
    df.join(dupe_ids_all_time, "account_id", "inner"),
    "account_id must be unique across all historical batches"
)

# 3. customer_ssn: not null
record_check(
    "customer_ssn_not_null",
    df.filter(F.col("customer_ssn").isNull()),
    "customer_ssn must not be null"
)

# 4. customer_ssn: unique — check against ALL history
dupe_ssns_all_time = (
    full_df.filter(F.col("customer_ssn").isNotNull())
           .groupBy("customer_ssn").count().filter("count > 1")
           .select("customer_ssn")
)
record_check(
    "customer_ssn_unique",
    df.join(dupe_ssns_all_time, "customer_ssn", "inner"),
    "customer_ssn must be unique across all historical batches"
)

# 5. customer_name: not null
record_check(
    "customer_name_not_null",
    df.filter(F.col("customer_name").isNull()),
    "customer_name must not be null"
)

# 6. balance: not null
record_check(
    "balance_not_null",
    df.filter(F.col("balance").isNull()),
    "balance must not be null"
)

# 7. balance >= 0
record_check(
    "balance_non_negative",
    df.filter(F.col("balance") < 0),
    "balance must be >= 0"
)

# 8. account_type: not null
record_check(
    "account_type_not_null",
    df.filter(F.col("account_type").isNull()),
    "account_type must not be null"
)

# 9. account_type: valid enum
valid_types = ["checking", "savings", "loan"]
record_check(
    "account_type_valid_enum",
    df.filter(~F.col("account_type").isin(valid_types)),
    "account_type must be one of checking, savings, loan"
)

In [0]:
results_df = spark.createDataFrame(results)
display(results_df)

overall_status = "FAIL" if any(r["status"] == "FAIL" for r in results) else "PASS"
print(f"\nOverall contract validation: {overall_status}")

In [0]:
import hashlib
import json
from datetime import datetime, timezone

# Hash the actual data content, not just the check results shape
row_strings = (
    df.select("account_id", "customer_ssn", "customer_name", "balance", "account_type")
      .selectExpr("concat_ws('|', account_id, customer_ssn, customer_name, balance, account_type) as row_str")
      .collect()
)
data_rows_str = "".join(sorted(r["row_str"] for r in row_strings))
data_hash = hashlib.sha256(data_rows_str.encode()).hexdigest()

# Force a fresh read of the certificates table to avoid caching issues
try:
    spark.sql("REFRESH TABLE banking_contracts_demo.certificates.contract_certificates")
except Exception:
    pass

try:
    prev_cert = spark.table("banking_contracts_demo.certificates.contract_certificates") \
        .orderBy(F.col("issued_at").desc()) \
        .limit(1) \
        .collect()
    prev_hash = prev_cert[0]["data_hash"] if prev_cert else "genesis"
except Exception:
    print("No previous certificate table found — this is expected on the first run.")
    prev_hash = "genesis"

certificate = {
    "contract_name": "accounts_daily",
    "contract_version": "v1",
    "batch_id": latest_batch_id,
    "issued_at": datetime.now(timezone.utc).isoformat(),
    "overall_status": overall_status,
    "total_records": total_records,
    "checks_passed": sum(1 for r in results if r["status"] == "PASS"),
    "checks_failed": sum(1 for r in results if r["status"] == "FAIL"),
    "check_details": json.dumps(results),
    "data_hash": data_hash,
    "prev_cert_hash": prev_hash,
}

cert_df = spark.createDataFrame([certificate])
cert_df.write.mode("append").saveAsTable("banking_contracts_demo.certificates.contract_certificates")

print(f"Certificate issued -- overall status: {overall_status}")
print(f"Batch: {latest_batch_id}")
print(f"Data hash: {data_hash[:16]}...")
print(f"Chained from: {prev_hash[:16] if prev_hash != 'genesis' else 'genesis'}...")

In [0]:
display(spark.table("banking_contracts_demo.certificates.contract_certificates").orderBy("issued_at"))